In [1]:
# Import required libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the feature-engineered dataset

input_file = "../data/tourism_experience_features.csv"

df = pd.read_csv(input_file)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (52930, 32)


In [3]:
# Check the columns required for recommendation

required_columns = [
    "Attraction",
    "CityName",
    "Country",
    "AttractionType",
    "Rating",
    "AttractionVisitCount",
    "AttractionAverageRating",
    "AttractionPopularity"
]

print("Required columns:")

for column in required_columns:
    print("-", column)

print("\nMissing required columns:")

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

print(missing_columns)

Required columns:
- Attraction
- CityName
- Country
- AttractionType
- Rating
- AttractionVisitCount
- AttractionAverageRating
- AttractionPopularity

Missing required columns:
[]


In [4]:
# Create an attraction-level dataset

attraction_df = (
    df.groupby(
        [
            "Attraction",
            "CityName",
            "Country",
            "AttractionType"
        ],
        dropna=False
    )
    .agg(
        VisitCount=("TransactionId", "count"),
        AverageRating=("Rating", "mean")
    )
    .reset_index()
)

print("Attraction-level dataset created!")
print("Shape:", attraction_df.shape)

display(attraction_df.head(10))

Attraction-level dataset created!
Shape: (1339, 6)


,Attraction,CityName,Country,AttractionType,VisitCount,AverageRating
0,Balekambang Beach,South Region,Australia,Beaches,1,5.000000
1,Balekambang Beach,South Region,Belgium,Beaches,1,4.000000
2,Balekambang Beach,South Region,Colombia,Beaches,1,5.000000
3,Balekambang Beach,South Region,France,Beaches,1,5.000000
4,Balekambang Beach,South Region,India,Beaches,1,5.000000
5,Balekambang Beach,South Region,Indonesia,Beaches,26,3.846154
6,Balekambang Beach,South Region,Malaysia,Beaches,1,5.000000
7,Balekambang Beach,South Region,Netherlands,Beaches,2,4.500000
8,Balekambang Beach,South Region,New Zealand,Beaches,1,4.000000
9,Balekambang Beach,South Region,United States,Beaches,1,3.000000


In [5]:
# Calculate a recommendation score
# Higher score means the attraction is more recommended

attraction_df["RecommendationScore"] = (
    0.6 * attraction_df["VisitCount"].rank(pct=True)
    +
    0.4 * attraction_df["AverageRating"].rank(pct=True)
)

# Sort attractions by recommendation score

attraction_df = attraction_df.sort_values(
    "RecommendationScore",
    ascending=False
).reset_index(drop=True)

display(attraction_df.head(10))

,Attraction,CityName,Country,AttractionType,VisitCount,AverageRating,RecommendationScore
0,Waterbom Bali,Douala,United Kingdom,Water Parks,585,4.702564,0.915609
1,Waterbom Bali,Douala,Australia,Water Parks,3523,4.652285,0.911576
2,Waterbom Bali,Douala,New Zealand,Water Parks,323,4.693498,0.907394
3,Waterbom Bali,Douala,Indonesia,Water Parks,463,4.628510,0.902016
4,Waterbom Bali,Douala,United States,Water Parks,250,4.684000,0.899253
5,Bromo Tengger Semeru National Park,South Region,Indonesia,National Parks,206,4.679612,0.893503
6,Waterbom Bali,Douala,Singapore,Water Parks,230,4.595652,0.882300
7,Waterbom Bali,Douala,India,Water Parks,133,4.593985,0.867438
8,Nusa Dua Beach,Douala,Indonesia,Beaches,181,4.530387,0.867214
9,Tanah Lot Temple,Douala,India,Religious Sites,419,4.463007,0.866468


In [7]:
# Function to recommend the top attractions

def recommend_attractions(top_n=10):
    
    recommendations = attraction_df[
        [
            "Attraction",
            "CityName",
            "Country",
            "AttractionType",
            "VisitCount",
            "AverageRating",
            "RecommendationScore"
        ]
    ].head(top_n)
    
    return recommendations

In [8]:
# Get the top 10 recommended attractions

recommendations = recommend_attractions(10)

display(recommendations)

,Attraction,CityName,Country,AttractionType,VisitCount,AverageRating,RecommendationScore
0,Waterbom Bali,Douala,United Kingdom,Water Parks,585,4.702564,0.915609
1,Waterbom Bali,Douala,Australia,Water Parks,3523,4.652285,0.911576
2,Waterbom Bali,Douala,New Zealand,Water Parks,323,4.693498,0.907394
3,Waterbom Bali,Douala,Indonesia,Water Parks,463,4.628510,0.902016
4,Waterbom Bali,Douala,United States,Water Parks,250,4.684000,0.899253
5,Bromo Tengger Semeru National Park,South Region,Indonesia,National Parks,206,4.679612,0.893503
6,Waterbom Bali,Douala,Singapore,Water Parks,230,4.595652,0.882300
7,Waterbom Bali,Douala,India,Water Parks,133,4.593985,0.867438
8,Nusa Dua Beach,Douala,Indonesia,Beaches,181,4.530387,0.867214
9,Tanah Lot Temple,Douala,India,Religious Sites,419,4.463007,0.866468


In [10]:
# Function to recommend attractions from a selected city

def recommend_by_city(city_name, top_n=10):
    
    city_recommendations = attraction_df[
        attraction_df["CityName"].str.lower() == city_name.lower()
    ]
    
    city_recommendations = city_recommendations[
        [
            "Attraction",
            "CityName",
            "Country",
            "AttractionType",
            "VisitCount",
            "AverageRating",
            "RecommendationScore"
        ]
    ].head(top_n)
    
    return city_recommendations

In [11]:
# Display some available cities

available_cities = (
    attraction_df["CityName"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print("Number of available cities:", len(available_cities))

display(available_cities.head(20))

Number of available cities: 3


0           Douala
20       N'Djamena
5     South Region
Name: CityName, dtype: str

In [12]:
# Test recommendation for the first available city

test_city = available_cities.iloc[0]

print("Recommendations for:", test_city)

city_recommendations = recommend_by_city(
    test_city,
    top_n=10
)

display(city_recommendations)

Recommendations for: Douala


,Attraction,CityName,Country,AttractionType,VisitCount,AverageRating,RecommendationScore
0,Waterbom Bali,Douala,United Kingdom,Water Parks,585,4.702564,0.915609
1,Waterbom Bali,Douala,Australia,Water Parks,3523,4.652285,0.911576
2,Waterbom Bali,Douala,New Zealand,Water Parks,323,4.693498,0.907394
3,Waterbom Bali,Douala,Indonesia,Water Parks,463,4.628510,0.902016
4,Waterbom Bali,Douala,United States,Water Parks,250,4.684000,0.899253
6,Waterbom Bali,Douala,Singapore,Water Parks,230,4.595652,0.882300
7,Waterbom Bali,Douala,India,Water Parks,133,4.593985,0.867438
8,Nusa Dua Beach,Douala,Indonesia,Beaches,181,4.530387,0.867214
9,Tanah Lot Temple,Douala,India,Religious Sites,419,4.463007,0.866468
10,Sacred Monkey Forest Sanctuary,Douala,Canada,Nature & Wildlife Areas,614,4.408795,0.865123


In [13]:
# Save the recommendation dataset

output_file = "../data/attraction_recommendations.csv"

attraction_df.to_csv(
    output_file,
    index=False
)

print("Recommendation dataset saved successfully!")
print("File:", output_file)
print("Shape:", attraction_df.shape)

Recommendation dataset saved successfully!
File: ../data/attraction_recommendations.csv
Shape: (1339, 7)


In [15]:
# Verify the saved recommendation dataset

recommendation_check = pd.read_csv(
    "../data/attraction_recommendations.csv"
)

print("Recommendation dataset loaded successfully!")
print("Shape:", recommendation_check.shape)

display(recommendation_check.head(10))

print("\nRecommendation System completed successfully! ")

Recommendation dataset loaded successfully!
Shape: (1339, 7)


,Attraction,CityName,Country,AttractionType,VisitCount,AverageRating,RecommendationScore
0,Waterbom Bali,Douala,United Kingdom,Water Parks,585,4.702564,0.915609
1,Waterbom Bali,Douala,Australia,Water Parks,3523,4.652285,0.911576
2,Waterbom Bali,Douala,New Zealand,Water Parks,323,4.693498,0.907394
3,Waterbom Bali,Douala,Indonesia,Water Parks,463,4.628510,0.902016
4,Waterbom Bali,Douala,United States,Water Parks,250,4.684000,0.899253
5,Bromo Tengger Semeru National Park,South Region,Indonesia,National Parks,206,4.679612,0.893503
6,Waterbom Bali,Douala,Singapore,Water Parks,230,4.595652,0.882300
7,Waterbom Bali,Douala,India,Water Parks,133,4.593985,0.867438
8,Nusa Dua Beach,Douala,Indonesia,Beaches,181,4.530387,0.867214
9,Tanah Lot Temple,Douala,India,Religious Sites,419,4.463007,0.866468



Recommendation System completed successfully! 
